<!-- SPDX-License-Identifier: GPL-2.0-only -->
# 30 POV export

The notebook judges the export of a scene as a scene file of POV-Ray. The coordinates of the lesson
are right handed with z up and those of POV-Ray left handed with y up, and `pov_p` of `POV.bas` maps
a point (x, y, z) to <y, z, -x>. The notebook checks that mapping, and checks numerically that the
scaling, the turns and the move POV-Ray reads put every corner of a body where the views draw it. It
compares the classic head and a golden scene with the file `ausgabe.pov` that version 1.3 wrote,
which it reads where it lies at the root of the repository, and checks the new body of the thesis
and the textures that `textures.inc` declares as pigments. A cell that proves an entry of the
register `Documentation/quirks.md` carries the tag its column Test names, and its checks name the ID
of the entry.

In [ ]:
render_with_povray = True
engine_binary = None  # the path of bausteine-graphql, the debug build by default

In [ ]:
import sys, json, math
sys.path.insert(0, '..')
from harness import *

In [ ]:
build_devtool()
import re, shutil

## 1. The room of `Def_Pov`

In [ ]:
text = devtool('pov', packed())
for line in ['#include "colors.inc"', '#include "textures.inc"', 'sky_sphere { pigment { Bright_Blue_Sky } }',
             'camera { location <15, 8, -15> look_at <0, 0, 0> }', 'light_source { <5, 10, -20> color White }',
             'plane { <0, 1, 0>, 0 pigment { checker color White color Silver }']:
    check(line in text, f'the header holds {line}')
custom = devtool('pov', packed(), '1,2,3', '4,5,6')
check('camera { location <2, 3, -1> look_at <5, 6, -4> }' in custom, 'a camera of the lesson is mapped to POV-Ray')

## 2. The mapping of points, sizes and turns

A point (x, y, z) becomes <y, z, -x>, a size (a, b, c) becomes the scaling <b, c, a>, and a turn
about z by an angle becomes `rotate <0, -angle, 0>`, about y `rotate <-angle, 0, 0>` and about x
`rotate <0, 0, angle>`, written z first as the matrix of an item applies them. The move is the place
of the item alone, since 2.0 has no point `P00` that 1.3 adds scaled.

In [ ]:
t = devtool('pov', packed(item('SPHERE', (1, 2, 3), (1.5, 1.5, 1.5))))
check('translate <2, 3, -1>' in t, 'T-p70a the centre (1, 2, 3) is translated to <2, 3, -1>')
t = devtool('pov', packed(item('CUBOID', size=(1, 2, 3))))
check('scale <2, 3, 1>' in t, 'T-p70a the edges (1, 2, 3) scale by <2, 3, 1>')
t = devtool('pov', packed(item('CUBE', rotation=(10, 20, 30))))
check('rotate <0, -30, 0> rotate <-20, 0, 0> rotate <0, 0, 10>' in t, 'T-p70a the turns are written z, y, x with their mapped signs')
for kind in ('CUBOID', 'PYRAMID', 'PRISM'):
    t = devtool('pov', packed(item(kind, (1, 2, 3), (2, 3, 4))))
    check('P00' not in t and t.split('// Objekt 1')[1].rstrip().splitlines()[-3].endswith('scale <3, 4, 2> translate <2, 3, -1>'),
          f'T-p70a a {kind.lower()} at (1, 2, 3) of the edges (2, 3, 4) moves by <2, 3, -1> alone, without P00', t.split('// Objekt 1')[1])

## 3. Every corner lands where the views draw it

POV-Ray applies `scale`, then each `rotate <a, b, c>` as a turn about x, then y, then z, each with
the usual matrix of a right handed turn of the coordinates, and `translate` last. The notebook
applies the statements of the export to the unit corners of POV-Ray and compares them with the
corners of the lesson moved by the matrix of the item and mapped by <y, z, -x>. Views, ray tracer
and export share the order of the thesis, z, then y, then x, and the export writes one `rotate` per
axis, where 1.3 writes one `rotate <-ry, -rz, rx>` that POV-Ray applies in another order.

In [ ]:
def rot3(axis, deg):
    c, s = math.cos(math.radians(deg)), math.sin(math.radians(deg))
    return {'x': [[1, 0, 0], [0, c, -s], [0, s, c]],
            'y': [[c, 0, s], [0, 1, 0], [-s, 0, c]],
            'z': [[c, -s, 0], [s, c, 0], [0, 0, 1]]}[axis]
def apply(m, p):
    return [sum(m[i][k] * p[k] for k in range(3)) for i in range(3)]
def vec(text):
    return [float(v) for v in text.split(',')]

def pov_transform(block, p):
    for name, args in re.findall(r'(scale|rotate|translate) <([^>]*)>', block):
        v = vec(args)
        if name == 'scale':
            p = [p[0] * v[0], p[1] * v[1], p[2] * v[2]]
        elif name == 'rotate':
            for axis, angle in zip('xyz', v):
                if angle:
                    p = apply(rot3(axis, angle), p)
        else:
            p = [p[0] + v[0], p[1] + v[1], p[2] + v[2]]
    return p

def lesson_to_pov(p):
    return [p[1], p[2], -p[0]]

def world(matrix16, p):
    m = [[matrix16[c * 4 + r] for c in range(4)] for r in range(4)]
    return [sum(m[i][k] * p[k] for k in range(3)) + m[i][3] for i in range(3)]

corners = [(x, y, z) for x in (0, 1) for y in (0, 1) for z in (0, 1)]
for rotation in [(0, 0, 0), (30, 0, 0), (0, 40, 0), (0, 0, 50), (15, -35, 70), (90, 90, 90)]:
    one = packed(item('CUBOID', (1, -2, 3), (2, 3, 4), rotation))
    block = devtool('pov', one).split('// Objekt 1')[1]
    m = devtool('matrix', one, 0)
    worst = 0.0
    for c in corners:
        # The unit box of POV-Ray is the image of the unit cube of the lesson.
        got = pov_transform(block.split('box {')[1], lesson_to_pov(c))
        want = lesson_to_pov(world(m, c))
        worst = max(worst, max(abs(a - b) for a, b in zip(got, want)))
    check(worst < 1e-3, f'F4-Q65 with the turns {rotation} every corner lands where the views draw it', worst)
    turns = [vec(a) for a in re.findall(r'rotate <([^>]*)>', block)]
    check(len(turns) == sum(1 for a in rotation if a) and all(sum(1 for v in t if v) == 1 for t in turns),
          f'F4-Q65 the export writes one rotate for each axis that the turns {rotation} turn', turns)

## 4. The groups of the CSG and the textures

A group becomes a nested `union`, `difference`, `intersection` or `merge`. A textured group gives
its texture to its children, which then write none of their own, and an empty group is left out,
since POV-Ray takes no empty CSG. A body has a colour or a texture, and the texture wins. A group
writes its own transformation after its members, its colour has no effect, and the members keep
theirs.

In [ ]:
werk = devtool('example', 'werkstueck')
t = devtool('pov', ','.join(map(str, werk['packed'])))
check(re.search(r'difference \{\s*// Objekt 2: Quader\s*box', t) is not None, 'the work piece is a difference that opens with its blank, a cuboid')
check(t.count('texture { Polished_Chrome }') == 1 and 'pigment' not in t.split('difference')[1], 'the texture of the group stands once and the children carry none')
t = devtool('pov', packed(item('INTERSECTION'), item('SPHERE', parent=0), item('MERGE')))
check('intersection {' in t and 'merge {' not in t and 'Leere Gruppe' in t, 'an empty group is a comment')
t = devtool('pov', packed(item('SPHERE', color=(1, 0, 0), texture=2)))
check('texture { Gold_Metal }' in t and 'pigment' not in t.split('// Objekt 1')[1], 'T-p93b a texture is named as POV-Ray knows it and wins over the colour')
t = devtool('pov', packed(item('SPHERE', color=(1, 0, 0))))
check('pigment { rgb <1, 0, 0> }' in t and 'texture {' not in t.split('// Objekt 1')[1], 'T-p93b without a texture, the Keine of 1.3, the body wears its colour again')
group = packed(item('UNION', (10, 0, 0), rotation=(0, 0, 90), color=(1, 0, 0)), item('SPHERE', (1, 0, 0), color=(0, 1, 1), parent=0), item('CUBE', color=(1, 1, 1), parent=0))
t = devtool('pov', group)
union = t[t.index('union {'):]
members = union.rindex('// Objekt 3')
check(union.index('scale <1, 1, 1> rotate <0, -90, 0> translate <0, 0, -10>') > members,
      'F4-Q100 a group writes its own transformation after its members', union)
check('rgb <0, 1, 1>' in union and 'rgb <1, 1, 1>' in union and 'rgb <1, 0, 0>' not in t,
      'T-p35i the members keep their colours, white among them, and the colour of the group has no effect', union)

## 5. New bodies

A declaration becomes `#declare` ahead of every item, the inner one first, since POV-Ray reads a
name before its use, and is placed once where it stands with `object { }`, as `Neu_Ende` does. An
instance is one more `object { }` with its own transformation. A name becomes an identifier of
POV-Ray.

In [ ]:
scene = packed(
    item('INSTANCE', position=(5, 0, 0), source=1),
    item('DECLARE'),
    item('CUBE', parent=1),
    item('DECLARE', parent=1),
    item('SPHERE', parent=3),
)
t = devtool('pov', scene)
first, second = t.index('#declare Neu_Neu_2 ='), t.index('#declare Neu_Neu =')
check(first < second < t.index('// Objekt 1:'), 'the inner declaration comes first and both precede every item')
check(len(re.findall(r'^object [{] Neu_Neu$', t, re.M)) == 2 and 'object { Neu_Neu_2' in t, 'the declaration is placed where it stands and by the instance')
check('translate <0, 0, -5>' in t.split('// Objekt 1:')[1].split('// Objekt 2')[0], 'the instance carries its own move')

## 6. The other bodies

The pyramid is a prism of the conic sweep, written centred, flipped and moved onto the unit square,
so its apex stands above the centre of its base and a turn moves it as the views draw it, with no
point `P00`. Commas separate the points of the pyramid and the prism, the disc keeps the normal <0,
1, 0> and its hole as a ratio clamped to 0.99, the cone the radius 0 at its apex, and every kind of
item has an export.

In [ ]:
t = devtool('pov', packed(item('DISC', size=(2, 2, 1), inner=0.5)))
check('disc { <0, 0, 0>, <0, 1, 0>, 1, 0.25' in t, 'M-39 a disc writes the normal <0, 1, 0> and its hole relative to its radius')
t = devtool('pov', packed(item('DISC', size=(2, 2, 1), inner=3)))
check('disc { <0, 0, 0>, <0, 1, 0>, 1, 0.99' in t, 'M-39 a hole wider than the disc is clamped to the ratio 0.99')
t = devtool('pov', packed(item('PYRAMID')))
check('prism { conic_sweep' in t and 'scale <1, -1, 1> translate <0.5, 1, -0.5>' in t and 'P00' not in t, 'M-28 a pyramid is a conic prism turned upright, without P00')
one = packed(item('PYRAMID', (1, -2, 3), (2, 3, 4), (15, -35, 70)))
t = devtool('pov', one)
block = t.split('// Objekt 1')[1]
shape = [vec(v) for v in re.findall(r'<([^<>]*)>', block[:block.index('scale')])]
m = devtool('matrix', one, 0)
# The conic sweep of POV-Ray puts the apex at its origin and the shape at the height 1.
apex = pov_transform(block, [0, 0, 0])
base = sorted(tuple(round(c, 3) for c in pov_transform(block, [u, 1, v])) for u, v in shape[:4])
want_base = sorted(tuple(round(c, 3) for c in lesson_to_pov(world(m, p))) for p in [(0, 0, 0), (1, 0, 0), (0, 1, 0), (1, 1, 0)])
check(close(apex, lesson_to_pov(world(m, (0.5, 0.5, 1))), 1e-3) and close(base, want_base, 2e-3),
      'M-28 M-29 the apex of a turned pyramid stands above the centre of its base, and base and apex land where the views draw them', apex, base, want_base)
check(re.search(r'<[^<>]*>\s*,\s*<', block) is not None and len(shape) == 5, 'M-31 commas separate the points of the pyramid', block)
t = devtool('pov', packed(item('PRISM')))
check(re.search(r'4,\s*<0, 0>, <0, -1>, <0.866025, -0.5>, <0, 0>', t) is not None, 'M-31 commas separate the points of the prism', t)
t = devtool('pov', packed(item('CONE')))
check('cone { <0, 0, 0>, 1, <0, 1, 0>, 0' in t, 'T-p104a a cone has its apex up with the radius 0')
t = devtool('pov', packed(item('SPHERE', (0.1 + 0.2, -0.0, 1e-7))))
check('translate <0, 0, -0.3>' in t, 'numbers are written short, with a point and without -0')
statements, rasters = {}, []
for kind in CODES:
    if kind in ('UNION', 'DIFFERENCE', 'INTERSECTION', 'MERGE', 'DECLARE'):
        scene, number = packed(item(kind), item('SPHERE', parent=0)), 1
    elif kind == 'INSTANCE':
        scene, number = packed(item('DECLARE'), item('SPHERE', parent=0), item('INSTANCE', source=0)), 3
    else:
        scene, number = packed(item(kind)), 1
    t = devtool('pov', scene)
    statements[kind] = t[t.rindex(f'// Objekt {number}:'):].splitlines()[1].split()[0]
    rasters += [kind] if 'Raster' in t else []
check(statements == {'SPHERE': 'sphere', 'CONE': 'cone', 'CYLINDER': 'cylinder', 'CUBE': 'box', 'CUBOID': 'box', 'PYRAMID': 'prism',
                     'PRISM': 'prism', 'DISC': 'disc', 'UNION': 'union', 'DIFFERENCE': 'difference', 'INTERSECTION': 'intersection',
                     'MERGE': 'merge', 'DECLARE': 'object', 'INSTANCE': 'object'} and rasters == [],
      'M-33 every kind of item has an export, and the raster is no item', statements, rasters)

## 7. POV-Ray reads the file

Where POV-Ray is installed, it renders every example from its exported file without an error.

In [ ]:
povray = shutil.which('povray')
if render_with_povray and povray:
    import subprocess, tempfile
    for e in devtool('examples'):
        scene = devtool('pov', ','.join(map(str, devtool('example', e['key'])['packed'])))
        with tempfile.TemporaryDirectory() as d:
            path = f'{d}/scene.pov'
            open(path, 'w').write(scene)
            r = subprocess.run([povray, '-D', '+W80', '+H60', f'+I{path}', f'+O{d}/scene.png'], capture_output=True, text=True, timeout=300)
            check(r.returncode == 0, f"POV-Ray renders the example {e['key']}", r.stderr[-2000:])
else:
    print('POV-Ray is not installed, the render is skipped')

## 8. The head of version 1.3

`Def_Pov` of `POV.bas` writes each line of comment with `appCr`, which ends it with two line breaks:
a rule of 58 equals signs, the version, the line `POV 2 Datei` with the German `Date` and `Time` of
Visual Basic, `TT.MM.JJJJ` and `HH:MM:SS`, and the rule again. The flag `--classic` of the devtool,
the argument `classic` of the field `pov` of the API, writes that layout with the version line of
2.0, which names version 1.3 and its author without claiming the file is theirs, and the flags
`--date` and `--time` the date and the time the page hands in. Both heads go on with the six
includes the export needs, and `shapes.inc`, which nothing of 2.0 uses, is left out. The head has
the structure of the head of `ausgabe.pov` line by line, and only the version line differs.

In [ ]:
quader = packed(item('CUBOID', (2, 2, 0), (4, 7, 2), (0, 0, -30), color=(0, 1, 1)))
classic = devtool('pov', quader, '--classic', '--date', '06.03.2002', '--time', '21:03:31')
check(devtool('pov', quader, '--classic', '--date=06.03.2002', '--time=21:03:31') == classic, 'the flags take their value after a blank or after an equals sign')
rule = '//' + '=' * 58
version = '// 3D-Bausteine Version 2.0 (klassisch) nach 3D-Baukasten 1.3 (c) 2002 H.Kleber'
dated = '// POV 2 Datei Datum: 06.03.2002 Zeit: 21:03:31'
head = classic.split('\n')[:8]
check(head == [rule, '', version, '', dated, '', rule, ''], 'F4-Q73 M-26 the classic head is the rule, the version, the date and the rule, each with an empty line after it', head)
includes = re.findall(r'^#include "([^"]+)"$', classic, re.M)
check(includes == ['colors.inc', 'textures.inc', 'glass.inc', 'metals.inc', 'woods.inc', 'stones.inc'],
      'F4-Q73 the head goes on with the six includes, and shapes.inc is left out', includes)
golden = (ROOT / 'ausgabe.pov').read_text(encoding='cp1252').replace('\r\n', '\n').split('\n')
check([golden[i] for i in (0, 1, 3, 4, 5, 6, 7)] == [head[i] for i in (0, 1, 3, 4, 5, 6, 7)],
      'F4-Q73 the classic head has the structure of the head of ausgabe.pov line by line', golden[:8])
check(golden[2].rstrip() == '// 3D-Baukasten Version 1.3 (c) 2002 H.Kleber' and head[2] != golden[2], 'only the version line differs, where 1.3 named itself', golden[2])
modern = devtool('pov', quader, '--date', '06.03.2002', '--time', '21:03:31')
check(modern.split('#include', 1)[1] == classic.split('#include', 1)[1] and not modern.startswith(rule), 'the head of 2.0 differs in the comment alone')

## 9. A golden scene of version 1.3

The file `ausgabe.pov` at the root holds one cyan Quader that version 1.3 exported: the edges 4, 7
and 2, the turn of -30 degrees about z, the place (2, 2, 0) and the colour `#00ffff`. The notebook
compares the export of 2.0 with it by meaning, from the first line `// Objekt` on: the number and
the trimmed name of each object, since 1.3 writes `// Objekt: 1 Quader` padded to 15 characters and
2.0 writes `// Objekt 1: Quader`, the corners of the box as the least and the greatest value on each
axis, since POV-Ray sorts them and 1.3 writes `<0, 0, 0>, <1, 1, -1>`, the scale, the sum of the
turns, the move and the colour as numbers, since 1.3 writes them with blanks and without a leading
zero. The two closing braces of 1.3 on the line of the pigment and the brace of 2.0 on a line of its
own carry no meaning of their own. Version 1.3 truncates every number to six characters with
`Left$(Str$(x), 6)`, and 2.0 rounds it to four decimals and writes it with a point and without an
exponent, every keyword apart from the number before it.

In [ ]:
def vector(text):
    return [float(v) for v in text.split(',')]

def meaning(text):
    # The objects of a scene file from the first comment // Objekt on, as numbers.
    objects = []
    for block in re.split(r'^\s*(?=// Objekt)', text[text.index('// Objekt'):], flags=re.M):
        if not block.strip():
            continue
        head = re.match(r'// Objekt:?\s*(\d+):?\s*(.*)', block)
        corners = re.search(r'box\s*\{\s*<([^>]*)>\s*,\s*<([^>]*)>', block)
        one = {'number': int(head.group(1)), 'name': head.group(2).strip()}
        if corners:
            a, b = vector(corners.group(1)), vector(corners.group(2))
            one['box'] = ([min(x, y) for x, y in zip(a, b)], [max(x, y) for x, y in zip(a, b)])
        for name in ('scale', 'translate', 'rgb'):
            found = re.search(name + r'\s*<([^>]*)>', block)
            one[name] = vector(found.group(1)) if found else None
        turns = [vector(t) for t in re.findall(r'rotate\s*<([^>]*)>', block)]
        one['rotate'] = [sum(t[i] for t in turns) for i in range(3)]
        objects.append(one)
    return objects

ours = meaning(classic)
theirs = meaning((ROOT / 'ausgabe.pov').read_text(encoding='cp1252'))
print(theirs)
check(ours == theirs, 'T-p102b the cyan Quader of 2.0 means what ausgabe.pov of 1.3 says, object by object', ours, theirs)
raw = (ROOT / 'ausgabe.pov').read_text(encoding='cp1252')
check('// Objekt: 1 Quader         ' in raw and '// Objekt 1: Quader\n' in classic, 'F4-Q57 1.3 pads the name to 15 characters and 2.0 writes it trimmed')
t = devtool('pov', packed(item('SPHERE', (0.123456, 123456.5, 1.5e-7), color=(0.05, 0.5, 1))))
check('translate <123456.5, 0, -0.1235>' in t and 'rgb <0.05, 0.5, 1>' in t, 'M-23 numbers are rounded to four decimals and keep their digits from 100000 on, without an exponent', t.split('// Objekt 1')[1])
numbers = [v.strip() for text in (classic, t) for args in re.findall(r'(?:scale|rotate|translate|rgb) <([^>]*)>', text) for v in args.split(',')]
check(numbers and all(re.fullmatch(r'-?\d+(\.\d{1,4})?', v) for v in numbers), 'M-23 every number of a statement has at most four decimals, a point and no exponent', numbers)
glued = [m.group(0) for text in (classic, t) for m in re.finditer(r'[\w.](?:scale|rotate|translate|pigment|texture)\b', text)]
check(glued == [], 'T-p104i every keyword stands apart from the number before it', glued)

## 10. The new body of the thesis

The thesis builds on p. 95 the new body `#Neu Werkstück1` from a `CSG-Differenz` of a `Quader` and a
`Zylinder`, closed by `CSG-Ende` and `Neu_Ende`. The export declares it ahead of every item on a
line of its own as `#declare Neu_Werkstueck1 = union {`, the prefix `Neu_` and the name in letters,
digits and underscores with the umlaut spelled out, holding the difference with the blank first, and
places it where it stands with `object { Neu_Werkstueck1 }` and a scale of its own group. The
devtool packs no names, so the notebook asks the engine `bausteine-graphql`.

In [ ]:
service = Engine(engine_binary)
page95 = service.data('mutation { createScene(name: "Seite 95") { id } }')['createScene']['id']
def add(input):
    return service.data('mutation($s: ID!, $i: AddItemInput!) { addItem(scene: $s, input: $i) { id } }', {'s': page95, 'i': input})['addItem']['id']
neu = add({'kind': 'DECLARE', 'name': 'Werkstück1'})
diff = add({'kind': 'DIFFERENCE', 'name': 'CSG-Differenz', 'parent': neu})
add({'kind': 'CUBOID', 'name': 'Quader', 'parent': diff, 'size': {'x': 4, 'y': 4, 'z': 2}})
add({'kind': 'CYLINDER', 'name': 'Zylinder', 'parent': diff, 'position': {'x': 2, 'y': 2, 'z': -1}, 'size': {'x': 1, 'y': 1, 'z': 4}})
text = service.data('query($s: ID!) { scene(id: $s) { pov(classic: true, date: "06.03.2002", time: "21:03:31") } }', {'s': page95})['scene']['pov']
service.stop()

def block(text, start):
    # The text from `start` to the brace that closes the brace it opens.
    at = text.index(start)
    depth = 0
    for i in range(text.index('{', at), len(text)):
        depth += {'{': 1, '}': -1}.get(text[i], 0)
        if depth == 0:
            return text[at:i + 1]
    raise Failure(f'the block {start} is not closed')

check(re.search(r'^#declare Neu_Werkstueck1 = union \{$', text, re.M) is not None, 'M-35 M-36 the body is declared on a line of its own as Neu_Werkstueck1, a union')
declared = block(text, '#declare Neu_Werkstueck1')
inner = block(declared, 'difference {')
check(inner.index('box {') < inner.index('cylinder {'), 'the declaration holds the difference with the blank before the drill', inner)
check(text.index('#declare') < text.index('// Objekt 1:'), 'the declaration precedes every item')
placed = re.findall(r'^object \{ Neu_Werkstueck1$', text, re.M)
check(len(placed) == 1, 'the body is placed once where it stands', placed)
check('scale <0, 0, 0>' not in text and 'scale <1, 1, 1>' in block(text, 'object { Neu_Werkstueck1'), 'M-38 the placement takes the scale of its own group and never scale <0, 0, 0>')

## 11. The textures that are pigments

Twenty of the 43 names of the list of version 1.3 are pigments in `textures.inc`, the agates, the
granite, the marbles, `Jade`, the woods and `DMFWood1` to `DMFWood5`, `Candy_Cane`, `Peel`,
`Blue_Sky2` and `Blood_Sky`. POV-Ray reads a pigment inside a texture only as `pigment { }`, so the
export writes `texture { pigment { Jade } }` for those and `texture { Gold_Metal }` for the others,
and the list of names stays the list of 1.3. The example `kugeln` wears eight of the pigments.

In [ ]:
textures = devtool('textures')
pigments = [t['name'] for t in textures if t['pigment']]
check(len(textures) == 43 and pigments == ['Brown_Agate', 'Sapphire_Agate', 'Pink_Granite', 'White_Marble', 'Red_Marble', 'Jade',
      'Cherry_Wood', 'Tan_Wood', 'Pine_Wood', 'Dark_Wood', 'White_Wood', 'DMFWood1', 'DMFWood2', 'DMFWood3', 'DMFWood4',
      'DMFWood5', 'Candy_Cane', 'Peel', 'Blue_Sky2', 'Blood_Sky'], 'M-43 twenty of the 43 names are pigments of textures.inc', pigments)
wrong = []
for t in textures:
    text = devtool('pov', packed(item('SPHERE', texture=t['index'])))
    want = f"texture {{ pigment {{ {t['name']} }} }}" if t['pigment'] else f"texture {{ {t['name']} }}"
    if want not in text:
        wrong.append((t['name'], re.findall(r'texture \{[^\n]*', text)))
check(wrong == [], 'M-43 the export wraps exactly the twenty pigments in pigment { }, and names the other 23 as textures', wrong)
kugeln = devtool('pov', ','.join(map(str, devtool('example', 'kugeln')['packed'])))
check(kugeln.count('texture { pigment {') == 8 and kugeln.count('texture {') == 16, 'M-43 the example kugeln wears eight wrapped pigments among its sixteen textures', re.findall(r'texture \{[^\n]*', kugeln))